# Strands SDK Basics

## What You'll Learn

| # | Concept | File |
|---|---------|------|
| 1 | Hello Agent | hello_agent.py |
| 1a | Agent with Custom Model | agent_with_custom_model.py |
| 2 | Agent with Tool | agent_with_tool.py |
| 3 | Agent with System Prompt | agent_with_system_prompt.py |
| 4 | MCP Server and Client | mcp_server.py / mcp_client.py |
| 5 | Chatbot | chatbot.py |

---
## Concept 1: Hello Agent

It takes just 3 lines of code — import, create, invoke — to build and run an AI agent:

```python
from strands import Agent       # 1 - Import the Agent class from the Strands SDK
agent = Agent()                 # 2 - Create an agent instance (defaults to Bedrock Claude Sonnet)
response = agent("question")   # 3 - Invoke the agent with a prompt (natural language input)
```

💡 No tools, no model configuration — the agent uses its built-in knowledge to answer any question in natural language.

In [ ]:
# hello_agent.py - Simplest Agent
from strands import Agent                        # Import the Agent class from the Strands SDK

agent = Agent()                                  # Create an agent (defaults to Bedrock Claude Sonnet)
response = agent("What is Amazon RDS for SQL Server")  # Ask the agent a question in natural language
print(response)                                  # Print the agent's response

🔍 **What model is it using?** Run this to check:

In [ ]:
from strands import Agent
agent = Agent()
print(agent.model.config)

---
## Concept 1a: Agent with Custom Model

`Agent()` defaults to Amazon Bedrock — Claude Sonnet. To use a specific model, configure a `BedrockModel`.

💡 **Region note:** The examples below read the region from the `AWS_REGION` environment variable (already set for your environment), falling back to `us-west-2`.

⚠️ **Sonnet 4.5 note:** Claude Sonnet 4.5 accepts only one of `temperature` or `top_p` per request, not both.

In [ ]:
# agent_with_custom_model.py - Override Model
from strands import Agent                        # Import the Agent class
from strands.models import BedrockModel          # Import BedrockModel to configure a specific LLM
import os                                        # For reading the AWS_REGION environment variable

model = BedrockModel(                            # Create a model configuration
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Specify the Claude Sonnet 4.5 model
    region_name=os.getenv("AWS_REGION", "us-west-2"),  # AWS region where Bedrock is enabled
    temperature=0.9,                             # Lower = more focused, higher = more creative
    max_tokens=4096,                             # Maximum length of the response
)
agent = Agent(model=model)                       # Create an agent using the custom model defined above
response = agent("What is Amazon Bedrock?")      # Invoke the agent with a prompt
print(response)

---
## Concept 2: Agent with Tool

Agents become powerful when you give them tools — functions they can call to interact with the real world.

### How It Works
```
You ask a question
       ↓
Agent REASONS about what to do
       ↓
Agent CALLS the right tool
       ↓
Tool RETURNS a result
       ↓
Agent RESPONDS with the answer
```

🔑 This is the pattern you'll use throughout the workshop — every database diagnostic query becomes a `@tool` that agents can call.

In [ ]:
# agent_with_tool.py - Tools
from strands import Agent, tool                  # Import Agent and the @tool decorator
from strands.models import BedrockModel          # Import BedrockModel to configure the LLM
import os                                        # For reading the AWS_REGION environment variable

@tool                                            # This decorator turns the function into an agent tool
def get_word_count(text: str) -> int:            # Accepts a text string parameter, returns an integer
    """Counts the number of words in a given string.
    The AI reads this docstring to decide when to use the tool."""
    return len(text.split())                     # split() breaks text into a list of words, len() counts them

model = BedrockModel(                            # Create a model configuration
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
agent = Agent(model=model, tools=[get_word_count])  # Create an agent with a model and tool
response = agent("How many words are in the sentence 'AI agents are powerful'?")
print(response)                                  # The agent will call get_word_count and respond

---
## Concept 3: Agent with System Prompt

The `system_prompt` is an instruction that defines how the agent behaves — its role, tone, and rules.

🔑 In this workshop, each agent has a system prompt that defines its specialty — the Database Health Agent focuses on CPU and memory, the Security Audit Agent focuses on access patterns, and so on.

In [ ]:
# agent_with_system_prompt.py - System Prompt
from strands import Agent, tool                  # Import Agent and the @tool decorator
from strands.models import BedrockModel          # Import BedrockModel to configure the LLM
import os                                        # For reading the AWS_REGION environment variable

@tool                                            # This decorator turns the function into an agent tool
def get_word_count(text: str) -> int:            # Accepts a text string parameter, returns an integer
    """Counts the number of words in a given string.
    The AI reads this docstring to decide when to use the tool."""
    return len(text.split())                     # split() breaks text into a list of words, len() counts them

model = BedrockModel(                            # Create a model configuration
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)

agent = Agent(                                   # Create an agent with model, tools, and behavior
    model=model,
    tools=[get_word_count],
    system_prompt="""You are an assistant who helps me count words.

Rules:
- Respond only to word counting requests
- Always use the get_word_count tool to count words
- Politely decline any unrelated questions
- Keep responses short and friendly"""
)

response = agent("How many words are in 'Strands makes building agents easy'?")
print(response)

In [ ]:
# Test the system prompt boundary - this should be politely declined
response = agent("What is the capital of France?")
print(response)

---
## Concept 4: MCP Server and Client

The **Model Context Protocol (MCP)** lets you expose tools as a server that any agent can connect to.

💡 Tools from `@tool` and tools from MCP servers are interchangeable — the agent doesn't care where the tool comes from.

### MCP Server

Run the server in a separate terminal:
```bash
python mcp_server.py
```

In [ ]:
%%writefile mcp_server.py
from mcp.server import FastMCP                   # Import FastMCP to create an MCP tool server

mcp = FastMCP("Calculator Server")               # Initialize a named MCP server

@mcp.tool(description="Add two numbers together")  # Register a function as an MCP tool
def add(x: int, y: int) -> int:
    """Add two numbers and return the result."""
    return x + y

mcp.run(transport="streamable-http")             # Start the server on HTTP (default port 8000)

### Start the MCP Server (background)

Run this cell first — it starts the server in the background so the client can connect:

In [ ]:
import subprocess, time

# Start MCP server in background
server_process = subprocess.Popen(['python', 'mcp_server.py'])
time.sleep(3)  # Wait for server to start
print(f"✅ MCP Server started (PID: {server_process.pid})")

### MCP Client

Now run the client — it connects to the MCP server running above:

In [ ]:
# mcp_client.py - Connect an Agent to the MCP Server
from mcp.client.streamable_http import streamablehttp_client  # HTTP transport for MCP
from strands import Agent                        # Import the Agent class
from strands.models import BedrockModel          # Import BedrockModel to configure the LLM
from strands.tools.mcp.mcp_client import MCPClient  # MCP client to connect to the server
import os                                        # For reading the AWS_REGION environment variable

model = BedrockModel(                            # Create a model configuration
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)

def create_transport():                          # Factory function that creates the HTTP connection
    return streamablehttp_client("http://localhost:8000/mcp/")

mcp_client = MCPClient(create_transport)         # Create an MCP client with the transport

with mcp_client:                                 # Connect to the MCP server
    tools = mcp_client.list_tools_sync()         # Discover all tools the server exposes
    agent = Agent(model=model, tools=tools)       # Give the model and MCP tools to the agent
    response = agent("What is 125 plus 375?")    # The agent calls the remote 'add' tool
    print(response)

In [ ]:
# Stop the MCP server
server_process.terminate()
server_process.wait()
print("🛑 MCP Server stopped")

---
## Concept 5: Chatbot

Agents maintain conversation context across turns. Let's demonstrate multi-turn memory by sending multiple messages to the same agent.

🔑 The key insight: the same `agent` instance remembers previous turns automatically.

In [ ]:
# Chatbot - Multi-turn conversation demo
from strands import Agent, tool                  # Import Agent and the @tool decorator
from strands.models import BedrockModel          # Import BedrockModel to configure the LLM
import os                                        # For reading the AWS_REGION environment variable

@tool                                            # This decorator turns the function into an agent tool
def get_word_count(text: str) -> int:            # Accepts a text string parameter, returns an integer
    """Counts the number of words in a given string.
    The AI reads this docstring to decide when to use the tool."""
    return len(text.split())                     # split() breaks text into a list of words, len() counts them

model = BedrockModel(                            # Create a model configuration
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
agent = Agent(model=model, tools=[get_word_count])  # Create an agent with model and tool
print("✅ Chatbot agent created")

In [ ]:
# Turn 1: Ask a question
response = agent("My name is Alice. How many words are in 'hello world from Strands'?")
print(f"Bot: {response}")

In [ ]:
# Turn 2: Follow-up — the agent remembers context from Turn 1!
response = agent("What's my name? And count the words in 'agents remember context'")
print(f"Bot: {response}")

☝️ Notice the agent remembered your name from Turn 1 — that's multi-turn conversation context in action!

For a full interactive chatbot loop, run as a script:
```bash
python chatbot.py
```

---
## Summary

| Concept | What You Learned | What's Coming |
|---------|-----------------|---------------|
| Agent | Create an agent in 3 lines | You'll build 5 specialized agents |
| Tool | `@tool` decorator for functions | You'll create 69 tools for SQL Server diagnostics |
| System Prompt | Define agent role, tone, and rules | Each agent gets a specialized persona |
| MCP | Expose/consume tools over HTTP | Connect agents to any external tool server |
| Conversation | Multi-turn context | Agents will retain memory across sessions via AgentCore |

### What You Built

| File | Concept |
|------|--------|
| hello_agent.py | Simplest Agent |
| agent_with_custom_model.py | Override Model |
| agent_with_tool.py | Tools |
| agent_with_system_prompt.py | System Prompt |
| mcp_server.py / mcp_client.py | MCP |
| chatbot.py | Conversational |

### Clean Up

Run this cell to remove the generated script files:

In [ ]:
import os
files_to_clean = [
    "hello_agent.py",
    "agent_with_custom_model.py",
    "agent_with_tool.py",
    "agent_with_system_prompt.py",
    "mcp_server.py",
    "mcp_client.py",
    "chatbot.py"
]
for f in files_to_clean:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")
    else:
        print(f"Skipped {f} (not found)")